# Movie Recommendation System — Model Building

This notebook rebuilds the content-based movie recommendation data used by the Streamlit application.

**Method:** movie metadata preprocessing → text tags → CountVectorizer → cosine similarity.

The notebook creates two files in the project root:

- `movies.pkl` — processed movie table (`id`, `original_title`, `tags`)
- `similarity_compressed.pkl` — precomputed cosine-similarity matrix used by `app.py`


## 1. Import libraries

In [5]:
import ast
import pickle
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth", 120)

## 2. Locate the project and load the datasets
The code searches the current directory and its parent directories for `app.py`, so it works when the notebook is opened from either the project root or the `notebooks/` folder.

In [6]:
def find_project_root(start_path=Path.cwd()):
    for folder in [start_path, *start_path.parents]:
        if (folder / "app.py").exists():
            return folder
    raise FileNotFoundError(
        "Could not find app.py. Open this notebook from inside the project folder."
    )

PROJECT_DIR = find_project_root()
DATASET_DIR = PROJECT_DIR / "datasets"

movies_path = DATASET_DIR / "tmdb_5000_movies.csv"
credits_path = DATASET_DIR / "tmdb_5000_credits.csv"

if not movies_path.exists() or not credits_path.exists():
    raise FileNotFoundError(
        "Dataset CSV files not found. Put tmdb_5000_movies.csv and "
        "tmdb_5000_credits.csv inside the project's datasets/ folder."
    )

movies = pd.read_csv(movies_path)
credits = pd.read_csv(credits_path)

print("Project folder:", PROJECT_DIR)
print("Movies dataset:", movies.shape)
print("Credits dataset:", credits.shape)

Project folder: /Users/shubhamaggarwal/Downloads/Movie-Recommendation-System
Movies dataset: (4803, 20)
Credits dataset: (4803, 4)


## 3. Merge and select relevant columns
The two TMDB tables are merged using the movie title. The project keeps the movie ID, title, overview, genres, keywords, cast and crew.

In [7]:
movies = movies.merge(credits, on="title")
movies = movies[["id", "original_title", "overview", "genres", "keywords", "cast", "crew"]].copy()

print("Merged dataset shape:", movies.shape)
movies.head()

Merged dataset shape: (4809, 7)


,id,original_title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn bet...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 878, ""name"": ""...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"": 2964, ""name"": ""future""}, {""id"": 3386, ""name"": ""space war""}, {""id"": 33...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""credit_id"": ""5602a8a7c3a3685532001c9a"", ""gender"": 2, ""id"": 65731, ""nam...","[{""credit_id"": ""52fe48009251416c750aca23"", ""department"": ""Editing"", ""gender"": 0, ""id"": 1721, ""job"": ""Editor"", ""name""..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will T...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fantasy""}, {""id"": 28, ""name"": ""Action""}]","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""name"": ""drug abuse""}, {""id"": 911, ""name"": ""exotic island""}, {""id"": 1319,...","[{""cast_id"": 4, ""character"": ""Captain Jack Sparrow"", ""credit_id"": ""52fe4232c3a36847f800b50d"", ""gender"": 2, ""id"": 85,...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""department"": ""Camera"", ""gender"": 2, ""id"": 120, ""job"": ""Director of Photo..."
2,206647,Spectre,A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. While M battles politica...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 80, ""name"": ""Crime""}]","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name"": ""based on novel""}, {""id"": 4289, ""name"": ""secret agent""}, {""id"": 966...","[{""cast_id"": 1, ""character"": ""James Bond"", ""credit_id"": ""52fe4d22c3a368484e1d8d6b"", ""gender"": 2, ""id"": 8784, ""name"":...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""department"": ""Sound"", ""gender"": 2, ""id"": 153, ""job"": ""Original Music Com..."
3,49026,The Dark Knight Rises,"Following the death of District Attorney Harvey Dent, Batman assumes responsibility for Dent's crimes to protect the...","[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""name"": ""Crime""}, {""id"": 18, ""name"": ""Drama""}, {""id"": 53, ""name"": ""Thrille...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853, ""name"": ""crime fighter""}, {""id"": 949, ""name"": ""terrorist""}, {""id"": 13...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Batman"", ""credit_id"": ""52fe4781c3a36847f8139869"", ""gender"": 2, ""id"": 389...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""department"": ""Sound"", ""gender"": 2, ""id"": 947, ""job"": ""Original Music Com..."
4,49529,John Carter,"John Carter is a war-weary, former military captain who's inexplicably transported to the mysterious and exotic plan...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adventure""}, {""id"": 878, ""name"": ""Science Fiction""}]","[{""id"": 818, ""name"": ""based on novel""}, {""id"": 839, ""name"": ""mars""}, {""id"": 1456, ""name"": ""medallion""}, {""id"": 3801,...","[{""cast_id"": 5, ""character"": ""John Carter"", ""credit_id"": ""52fe479ac3a36847f813ea75"", ""gender"": 2, ""id"": 60900, ""name...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""department"": ""Writing"", ""gender"": 2, ""id"": 7, ""job"": ""Screenplay"", ""name..."


## 4. Clean missing and duplicate records
Rows with missing values are removed, matching the original project workflow.

In [8]:
print("Missing values before cleaning:")
display(movies.isnull().sum())

movies.dropna(inplace=True)
movies.drop_duplicates(inplace=True)

print("Dataset shape after cleaning:", movies.shape)
print("Remaining missing values:", int(movies.isnull().sum().sum()))

Missing values before cleaning:


id                0
original_title    0
overview          3
genres            0
keywords          0
cast              0
crew              0
dtype: int64

Dataset shape after cleaning: (4806, 7)
Remaining missing values: 0


## 5. Extract genres, keywords, cast and director
TMDB stores these fields as string representations of lists of dictionaries. `ast.literal_eval` safely converts those strings into Python data structures.

In [9]:
def extract_names(value):
    """Return the 'name' values from a TMDB list-of-dictionaries field."""
    return [item["name"] for item in ast.literal_eval(value)]

def extract_top_cast(value, limit=3):
    """Return up to the first three cast members."""
    return [item["name"] for item in ast.literal_eval(value)[:limit]]

def extract_director(value):
    """Return the first director found in the crew list."""
    crew = ast.literal_eval(value)
    for person in crew:
        if person.get("job") == "Director":
            return [person["name"]]
    return []

In [10]:
movies["genres"] = movies["genres"].apply(extract_names)
movies["keywords"] = movies["keywords"].apply(extract_names)
movies["cast"] = movies["cast"].apply(extract_top_cast)
movies["crew"] = movies["crew"].apply(extract_director)

movies[["original_title", "genres", "keywords", "cast", "crew"]].head()

,original_title,genres,keywords,cast,crew
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colony, society, space travel, futuristic, romance, space, alien, tribe, al...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india trading company, love of one's life, traitor, shipwreck, strong woman,...","[Johnny Depp, Orlando Bloom, Keira Knightley]",[Gore Verbinski]
2,Spectre,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi6, british secret service, united kingdom]","[Daniel Craig, Christoph Waltz, Léa Seydoux]",[Sam Mendes]
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret identity, burglar, hostage drama, time bomb, gotham city, vigilante, co...","[Christian Bale, Michael Caine, Gary Oldman]",[Christopher Nolan]
4,John Carter,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel, princess, alien, steampunk, martian, escape, edgar rice burroughs, a...","[Taylor Kitsch, Lynn Collins, Samantha Morton]",[Andrew Stanton]


## 6. Prepare the movie tags
The overview is split into words. Spaces are removed within names such as `Sam Worthington`, so the name is treated as one token (`SamWorthington`). All selected fields are then combined into a single `tags` column.

In [11]:
movies["overview"] = movies["overview"].apply(lambda text: text.split())

for column in ["genres", "keywords", "cast", "crew"]:
    movies[column] = movies[column].apply(
        lambda values: [value.replace(" ", "") for value in values]
    )

movies["tags"] = (
    movies["overview"]
    + movies["genres"]
    + movies["keywords"]
    + movies["cast"]
    + movies["crew"]
)

new_df = movies[["id", "original_title", "tags"]].copy()
new_df.head()

,id,original_title,tags
0,19995,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marine, is, dispatched, to, the, moon, Pandora, on, a, unique, mission,, bu..."
1,285,Pirates of the Caribbean: At World's End,"[Captain, Barbossa,, long, believed, to, be, dead,, has, come, back, to, life, and, is, headed, to, the, edge, of, t..."
2,206647,Spectre,"[A, cryptic, message, from, Bond’s, past, sends, him, on, a, trail, to, uncover, a, sinister, organization., While, ..."
3,49026,The Dark Knight Rises,"[Following, the, death, of, District, Attorney, Harvey, Dent,, Batman, assumes, responsibility, for, Dent's, crimes,..."
4,49529,John Carter,"[John, Carter, is, a, war-weary,, former, military, captain, who's, inexplicably, transported, to, the, mysterious, ..."


## 7. Apply stemming and normalize text
Porter stemming reduces related word forms to a common stem (for example, `loved` and `loving` can be reduced toward `love`). The tags are then converted into lowercase strings.

In [12]:
stemmer = PorterStemmer()

def stem_tags(tags):
    words = " ".join(tags).lower().split()
    return " ".join(stemmer.stem(word) for word in words)

new_df["tags"] = new_df["tags"].apply(stem_tags)

new_df.head()

,id,original_title,tags
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispatch to the moon pandora on a uniqu mission, but becom torn between fol..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha come back to life and is head to the edg of the earth with will turner ..."
2,206647,Spectre,a cryptic messag from bond’ past send him on a trail to uncov a sinist organization. while m battl polit forc to kee...
3,49026,The Dark Knight Rises,"follow the death of district attorney harvey dent, batman assum respons for dent' crime to protect the late attorney..."
4,49529,John Carter,"john carter is a war-weary, former militari captain who' inexplic transport to the mysteri and exot planet of barsoo..."


## 8. Convert text into numerical vectors
`CountVectorizer` applies a Bag-of-Words representation. It keeps at most 5,000 features and removes English stop words.

In [13]:
vectorizer = CountVectorizer(max_features=5000, stop_words="english")
vectors = vectorizer.fit_transform(new_df["tags"])

print("Vector matrix shape:", vectors.shape)

Vector matrix shape: (4806, 5000)


## 9. Calculate cosine similarity
Cosine similarity compares every movie vector with every other movie vector. The resulting square matrix contains one similarity score for each movie pair.

In [14]:
similarity = cosine_similarity(vectors)

print("Similarity matrix shape:", similarity.shape)

Similarity matrix shape: (4806, 4806)


## 10. Test the recommendation function
This uses the same approach as the Streamlit app: find the selected movie's row, sort its similarity scores in descending order, skip the movie itself, and return the next five titles.

In [15]:
def recommend(movie_title, number_of_recommendations=5):
    matches = new_df.index[new_df["original_title"] == movie_title].tolist()

    if not matches:
        raise ValueError(f"Movie not found: {movie_title}")

    movie_index = matches[0]
    distances = similarity[movie_index]

    ranked_movies = sorted(
        enumerate(distances),
        key=lambda item: item[1],
        reverse=True
    )

    # Skip the first result because it is the selected movie itself.
    recommendations = ranked_movies[1:number_of_recommendations + 1]

    return [
        (new_df.iloc[index]["original_title"], float(score))
        for index, score in recommendations
    ]

In [16]:
recommend("Avatar")

[('Aliens vs Predator: Requiem', 0.2867696673382022),
 ('Aliens', 0.26901379342448517),
 ('Falcon Rising', 0.2605130246476754),
 ('Independence Day', 0.255608593705383),
 ('Titan A.E.', 0.25038669783359574)]

## 11. Save the files used by the application
The filenames below match the cleaned Streamlit app. `joblib` compression keeps the similarity matrix in a compressed file.

In [17]:
movies_output = PROJECT_DIR / "movies.pkl"
similarity_output = PROJECT_DIR / "similarity_compressed.pkl"

with open(movies_output, "wb") as file:
    pickle.dump(new_df, file)

joblib.dump(similarity, similarity_output, compress=3)

print("Saved:", movies_output)
print("Saved:", similarity_output)

Saved: /Users/shubhamaggarwal/Downloads/Movie-Recommendation-System/movies.pkl
Saved: /Users/shubhamaggarwal/Downloads/Movie-Recommendation-System/similarity_compressed.pkl


## 12. Summary

The notebook has:

1. Loaded and merged the TMDB movie and credits datasets.
2. Extracted genres, keywords, the first three cast members and director.
3. Combined the selected metadata into text tags.
4. Applied stemming and lowercase normalization.
5. Converted tags into Bag-of-Words vectors using `CountVectorizer`.
6. Calculated pairwise cosine similarity.
7. Tested movie recommendations.
8. Saved `movies.pkl` and `similarity_compressed.pkl` for the Streamlit application.

**To run the web app after generating the files**, return to the project root and execute:

```bash
streamlit run app.py
```